# MIT News: repair duplicated/missing article bodies

The first-pass MIT scrape left 33 articles with duplicated or unusable bodies. This notebook:

1. detects those rows and exports their links,
2. re-scrapes them with `requests` + **Trafilatura** as a generic body extractor,
3. merges the recovered fields back by normalized URL.

Output: `./Data/MIT Articles merged.csv` — the final MIT corpus used by the analysis (copied to the project-level `Data/MIT Articles.csv`).


In [ ]:
import random
import re
import time
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from trafilatura import extract as trafilatura_extract
from urllib3.util.retry import Retry

DATA_DIR = Path("./Data")
MIT_PATH = DATA_DIR / "MIT Articles.csv"
DUP_LINKS_PATH = DATA_DIR / "MIT Duplicated Article Body Links.csv"
RECOVERED_PATH = DATA_DIR / "MIT Duplicated Article Body articles.csv"
MERGED_PATH = DATA_DIR / "MIT Articles merged.csv"

DATA_DIR.mkdir(parents=True, exist_ok=True)

mit_df = pd.read_csv(MIT_PATH)
print(f"MIT rows: {len(mit_df)}")

## 1. Detect duplicated bodies and export their links

In [ ]:
normalized_body = (
    mit_df["Article Body"].astype("string")
    .str.replace(r"\s+", " ", regex=True).str.strip()
)
normalized_body = normalized_body.mask(
    normalized_body.isna() | normalized_body.eq("")
    | normalized_body.str.lower().isin(["nan", "none", "null", "<na>"]),
    pd.NA,
)
duplicated_body_mask = normalized_body.notna() & normalized_body.duplicated(keep=False)

dup_links = (
    mit_df.loc[duplicated_body_mask, ["Article Header", "Url"]]
    .rename(columns={"Article Header": "Article", "Url": "Link"})
    .drop_duplicates().reset_index(drop=True)
)
print(f"Articles with duplicated bodies: {len(dup_links)}")
dup_links.to_csv(DUP_LINKS_PATH, index=False, encoding="utf-8-sig")
dup_links

## 2. Re-scrape the affected articles with Trafilatura

In [ ]:
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=4, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)))
session.headers.update({
    "User-Agent": ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0 Safari/537.36"),
    "Accept-Language": "en-US,en;q=0.9",
})

def clean_text(value):
    return re.sub(r"\s+", " ", str(value)).strip() if value is not None else ""

def get_meta(soup, selectors):
    for selector in selectors:
        node = soup.select_one(selector)
        if node and clean_text(node.get("content")):
            return clean_text(node.get("content"))
    return ""

def get_text(soup, selectors):
    for selector in selectors:
        node = soup.select_one(selector)
        if node and clean_text(node.get_text(" ", strip=True)):
            return clean_text(node.get_text(" ", strip=True))
    return ""

def extract_body_blocks(html, url):
    text = trafilatura_extract(
        html, url=url, include_comments=False, include_tables=False,
        include_links=False, include_images=False, favor_precision=True, output_format="txt",
    )
    if not text:
        return []
    blocks = []
    for line in text.splitlines():
        line = clean_text(line)
        if line and (not blocks or line != blocks[-1]):
            blocks.append(line)
    return blocks

In [ ]:
def rescrape_article(title, url):
    record = {"Published Date": "", "Author": "", "Source": "", "Article Header": title,
              "Sub_Headings": "", "Article Body": [], "Url": url,
              "Scrape Status": "Failed", "Error": ""}
    try:
        response = session.get(url, timeout=30, allow_redirects=True)
        record["Url"] = response.url
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "lxml")

        record["Article Header"] = (
            get_meta(soup, ['meta[property="og:title"]', 'meta[name="twitter:title"]'])
            or get_text(soup, ["h1"]) or title
        )
        record["Published Date"] = (
            get_meta(soup, ['meta[property="article:published_time"]', 'meta[name="date"]'])
            or get_text(soup, ["time", ".news-article--publication-date"])
        )
        record["Author"] = (
            get_meta(soup, ['meta[name="author"]'])
            or get_text(soup, [".news-article--author", ".byline", ".author"])
        )
        record["Source"] = get_text(soup, [".news-article--contact", ".article-source", ".source"])
        record["Sub_Headings"] = (
            get_meta(soup, ['meta[property="og:description"]', 'meta[name="description"]'])
            or get_text(soup, [".news-article--dek", ".dek"])
        )

        blocks = extract_body_blocks(response.text, response.url)
        metadata = {clean_text(record[k]) for k in
                    ["Article Header", "Sub_Headings", "Author", "Source", "Published Date"]} - {""}
        blocks = [b for b in blocks if clean_text(b) not in metadata]

        if sum(len(b) for b in blocks) < 200:
            raise ValueError("Extracted body is too short.")
        record["Article Body"] = blocks
        record["Scrape Status"] = "Success"
    except Exception as exc:
        record["Error"] = f"{type(exc).__name__}: {exc}"
    return record


results = []
for _, row in tqdm(dup_links.iterrows(), total=len(dup_links)):
    results.append(rescrape_article(row["Article"], row["Link"]))
    time.sleep(random.uniform(0.5, 1.2))

recovered_df = pd.DataFrame(results)
print(recovered_df["Scrape Status"].value_counts(dropna=False))
recovered_df[["Published Date", "Author", "Source", "Article Header",
              "Sub_Headings", "Article Body", "Url"]].to_csv(
    RECOVERED_PATH, index=False, encoding="utf-8-sig")
print(f"Saved to: {RECOVERED_PATH}")

## 3. Merge recovered rows back into the MIT corpus by URL

In [ ]:
def normalize_url(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip().split("#", 1)[0].split("?", 1)[0]
    value = value.removeprefix("https://").removeprefix("http://")
    return value.removeprefix("www.").rstrip("/").casefold() or pd.NA

def is_missing(value):
    if pd.isna(value):
        return True
    return str(value).strip().casefold() in {"", "nan", "none", "null", "<na>", "[]", "{}"}

recovered = pd.read_csv(RECOVERED_PATH)
recovered["_key"] = recovered["Url"].map(normalize_url)
recovered = recovered.dropna(subset=["_key"]).drop_duplicates(subset="_key", keep="last")

merged = mit_df.copy()
merged["_key"] = merged["Url"].map(normalize_url)

update_columns = ["Published Date", "Author", "Source", "Article Header",
                  "Sub_Headings", "Article Body", "Url"]
updated_fields = 0
for _, rec in recovered.iterrows():
    idx = merged.index[merged["_key"].eq(rec["_key"])]
    for i in idx:
        for col in update_columns:
            if not is_missing(rec.get(col)) and str(merged.at[i, col]).strip() != str(rec[col]).strip():
                merged.at[i, col] = rec[col]
                updated_fields += 1

merged = merged.drop(columns="_key")
problem_before = int(mit_df["Article Body"].map(is_missing).sum())
problem_after = int(merged["Article Body"].map(is_missing).sum())
print(f"Fields updated: {updated_fields}")
print(f"Problematic bodies before: {problem_before} | after: {problem_after}")

merged.to_csv(MERGED_PATH, index=False, encoding="utf-8-sig")
print(f"Saved final MIT corpus to: {MERGED_PATH}")